## 初步构建

In [1]:
# ========================
# Step 1: 挂载 Google Drive
# ========================
from google.colab import drive
drive.mount('/content/drive')

# ========================
# Step 2: 克隆项目
# ========================
# 替换为你的 GitHub 仓库地址
!git clone https://github.com/codeGease-1/CrossModal-Defog-OVOD.git

# ========================
# Step 3: 进入项目目录
# ========================
%cd /content/CrossModal-Defog-OVOD

# ========================
# Step 4: 安装依赖
# ========================
!pip install -r requirements.txt

# ========================
# Step 5: 设置 PYTHONPATH
# ========================
import sys
sys.path.insert(0, '/content/CrossModal-Defog-OVOD')

# ========================
# Step 6: 检查 GPU 环境
# ========================
!nvidia-smi

# 检查 PyTorch GPU 支持
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
      if torch.cuda.is_available() else "No GPU")


Mounted at /content/drive
Cloning into 'CrossModal-Defog-OVOD'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 339 (delta 176), reused 251 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (339/339), 214.96 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (176/176), done.
/content/CrossModal-Defog-OVOD
Sun Sep 27 06:50:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M.

## 阶段1：

### 运行训练：

In [ ]:
# 基础——使用默认配置训练
!python scripts/train/train_haze_density.py --config configs/haze_density.yaml

雾密度感知网络训练
配置：configs/haze_density.yaml
设备：cuda
Batch Size: 4
图像尺寸：256
学习率：1e-4
训练轮数：100

训练逻辑待实现...
请等待后续阶段完成模型和数据模块的实现。


In [ ]:
# 自定义——修改 batch_size 和 learning_rate
!python scripts/train/train_haze_density.py \
    --config configs/haze_density.yaml \
    --data.batch_size 8 \
    --train.lr 5e-4

usage: train_haze_density.py [-h] [--config CONFIG] [--resume RESUME]
                             [--data_batch_size DATA_BATCH_SIZE]
                             [--data_image_size DATA_IMAGE_SIZE]
                             [--train_lr TRAIN_LR]
                             [--train_epochs TRAIN_EPOCHS]
                             [--model_base_channels MODEL_BASE_CHANNELS]
train_haze_density.py: error: unrecognized arguments: --data.batch_size 8 --train.lr 5e-4


In [ ]:
# 断点——从 checkpoint 继续训练
!python scripts/train/train_haze_density.py \
    --config configs/haze_density.yaml \
    --resume experiments/haze_density/checkpoints/latest.pt

雾密度感知网络训练
配置：configs/haze_density.yaml
设备：cuda
Batch Size: 4
图像尺寸：256
学习率：1e-4
训练轮数：100

训练逻辑待实现...
请等待后续阶段完成模型和数据模块的实现。


### SmokeTest:

In [ ]:
# 形状测试（不训练，仅验证 forward/backward）
!python scripts/utils/smoke_test.py


CrossModal-Defog-OVOD - Smoke Test

1. PyTorch 环境检查
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

使用设备：cuda

2. 物理先验模块测试
待实现：physical_priors 模块

3. 模型构建测试
待实现：haze_density_net 模块

4. Forward/Backward 测试
待实现：forward/backward 测试

5. 形状一致性测试
待实现：shape consistency 测试

测试结果汇总
物理先验: ✓
模型构建: ✓
Forward/Backward: ✓
形状一致性: ✓

✓ 所有测试通过！


## 阶段2：物理先验测试和可视化：

In [ ]:
## pull更新
%cd /content/CrossModal-Defog-OVOD
## !git checkout -- scripts/train/train_haze_density.py
!git pull

/content/CrossModal-Defog-OVOD
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 515 bytes | 515.00 KiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   b22ca05..fb77ca2  main       -> origin/main
Updating b22ca05..fb77ca2
Fast-forward
 scripts/test_physical_prior.py | 25 +++++++++++++++++++++++--
 1 file changed, 23 insertions(+), 2 deletions(-)


In [ ]:
# Step 3: 运行物理先验测试
!python scripts/test/test_physical_prior.py
# Step 4: 运行可视化（可选）
!python scripts/data/visualize_physical_prior.py --generate-test


物理先验模块测试

GPU available: Tesla T4

7. Constants Test (申报书规定参数)
  [OK] WEIGHT_DARK = 0.5 (expected 0.5)
  [OK] WEIGHT_CONTRAST = 0.3 (expected 0.3)
  [OK] WEIGHT_COLOR = 0.2 (expected 0.2)
  [OK] EXPONENT_MU = 1.5 (expected 1.5)
[OK] All constants match 申报书规定!

1. Shape Test
  [OK] dark_channel: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] local_contrast: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] color_shift: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] d_hat: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] c_hat: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] k_hat: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] s_hat: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] s_final: [1, 3, 256, 256] -> [1, 1, 256, 256]
  [OK] dark_channel: [2, 3, 128, 128] -> [2, 1, 128, 128]
  [OK] local_contrast: [2, 3, 128, 128] -> [2, 1, 128, 128]
  [OK] color_shift: [2, 3, 128, 128] -> [2, 1, 128, 128]
  [OK] d_hat: [2, 3, 128, 128] -> [2, 1, 128, 128]
  [OK] c_hat: [2, 3, 128, 128] -> [2, 1, 128, 128]
  [OK] k_hat: [

## 阶段3：实现雾密度网络核心特征提取模块

In [ ]:
# Step 1: 更新代码
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 458 bytes | 458.00 KiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   49d470d..01abc78  main       -> origin/main
Updating 49d470d..01abc78
Fast-forward
 scripts/test_core_modules.py | 10 +++++-----
 1 file changed, 5 insertions(+), 5 deletions(-)


In [ ]:
# Step 2: 运行核心模块测试
!python scripts/test/test_core_modules.py


雾密度网络核心模块测试

GPU available: Tesla T4

1. Encoder Test
  [OK] Shape: (2, 3, 256, 256) -> (2, 64, 128, 128) (expected (2, 64, 128, 128))
  [OK] dtype: torch.float32
  [OK] device: cpu
  [OK] finite: no NaN/Inf
  [OK] backward: gradient computed
  Parameters: 28,512
[OK] Encoder test passed!

2. ResidualBlock Test
  [OK] Shape: (2, 64, 128, 128) -> (2, 64, 128, 128)
  [OK] dtype: torch.float32
  [OK] device: cpu
  [OK] finite: no NaN/Inf
  [OK] backward: gradient computed
  Parameters: 73,728
[OK] ResidualBlock test passed!

3. DilatedResidualBlock Test

  Testing dilation=2
    [OK] Shape: (2, 64, 128, 128) -> (2, 64, 128, 128)
    [OK] finite: no NaN/Inf
    [OK] backward: gradient computed

  Testing dilation=3
    [OK] Shape: (2, 64, 128, 128) -> (2, 64, 128, 128)
    [OK] finite: no NaN/Inf
    [OK] backward: gradient computed

  Testing dilation=4
    [OK] Shape: (2, 64, 128, 128) -> (2, 64, 128, 128)
    [OK] finite: no NaN/Inf
    [OK] backward: gradient computed

[OK] DilatedRes

## 阶段4：完成完整雾密度感知网络

In [ ]:
# Step 1: 更新代码
!git pull

Already up to date.


In [ ]:
# Step 2: 运行完整模型测试
!python scripts/test/test_model.py


雾密度感知网络完整测试

GPU available: Tesla T4
PyTorch version: 2.11.0+cu128

1. Shape Test
  [OK] (1, 3, 256, 256) -> (1, 1, 256, 256) (expected (1, 1, 256, 256))
  [OK] (2, 3, 128, 128) -> (2, 1, 128, 128) (expected (2, 1, 128, 128))
  [OK] (4, 3, 512, 512) -> (4, 1, 512, 512) (expected (4, 1, 512, 512))
[OK] Shape test passed!

2. Range Test
  [OK] Output range: [0.500000, 0.986497]
[OK] Range test passed!

3. Finite Test
  [OK] No NaN/Inf in output
[OK] Finite test passed!

4. Forward Test
  [OK] Forward completed: (2, 3, 256, 256) -> (2, 1, 256, 256)
[OK] Forward test passed!

5. Backward Test
  [OK] Backward completed, gradient computed
[OK] Backward test passed!

6. GPU Test
  GPU: Tesla T4
  [OK] Output on GPU: cuda:0
[OK] GPU test passed!

7. Performance Test
  GPU: Tesla T4

  Parameters: 864,752

  Parameter breakdown:
    Encoder:    28,512 (3.3%)
    MultiScale: 700,416 (81.0%)
    Fusion:     122,880 (14.2%)
    Decoder:    12,944 (1.5%)

  Forward time: 13.41 ms (avg over 10 runs

## 阶段 5A.5 **数据集实际获取与核验准备**（必做）

In [ ]:
# Step 1: 更新代码
!git pull

Already up to date.


### 推荐方案：通过 Google Drive 加载大文件
1. 手动将 `RSHaze+.zip` 上传到你的 Google Drive 根目录。
2. 运行以下代码挂载 Drive 并将其移动到 Colab 运行环境。

In [2]:
#from google.colab import drive
import os

# 1. 挂载 Drive
drive.mount('/content/drive')

# 2. 定义路径 (假设文件在 Drive 根目录)
drive_path = '/content/drive/MyDrive/dataset/RSHaze+.zip'
local_path = '/content/datasets/RSHaze+.zip'

os.makedirs('/content/datasets', exist_ok=True)

# 3. 使用 cp 命令拷贝（比网页上传快且稳）
if os.path.exists(drive_path):
    print("正在从 Drive 拷贝大文件...")
    !cp "{drive_path}" "{local_path}"
    print("拷贝完成，开始解压...")
    !unzip -q "{local_path}" -d /content/datasets/RSHazePlus
    print("解压完成！")
else:
    print(f"错误：在 Drive 中未找到文件 {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在从 Drive 拷贝大文件...
拷贝完成，开始解压...
解压完成！


In [ ]:
!rm -rf /content/datasets/RSHazePlus/

In [ ]:
# 创建目录
#!mkdir -p /content/datasets

# 下载 RSHaze+ (主训练集)
#!wget https://zenodo.org/records/13837162/files/RSHaze+.zip?download=1 -O /content/datasets/RSHaze+.zip
#!unzip -q /content/datasets/RSHaze+.zip -d /content/datasets/RSHazePlus
#!rm /content/datasets/RSHaze+.zip

# 下载 RS-Haze (补充训练集)
!git clone https://github.com/zhuqinghe/RS-Haze.git /content/datasets/RSHaze

# 下载 RRSHID (外部测试集)
!wget https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip -O /content/datasets/RRSHID.zip
!unzip -q /content/datasets/RRSHID.zip -d /content/datasets/RRSHID
!rm /content/datasets/RRSHID.zip

--2026-08-28 15:11:17--  https://zenodo.org/records/13837162/files/RSHaze+.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.184.103.118, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7380255872 (6.9G) [application/octet-stream]
Saving to: ‘/content/datasets/RSHaze+.zip’

RSHaze+.zip          93%[=================>  ]   6.39G  2.59MB/s    eta 2m 45s ^C
[/content/datasets/RSHaze+.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/datasets/RSHaze+.zip or
        /content/datasets/RSHaze+.zip.zip, and cannot find /content/datasets/RSHaze+.zip.ZIP, period.


In [ ]:
!python scripts/data/inspect_dataset.py --dataset RSHazePlus --data_dir /content/datasets/RSHazePlus
!python scripts/data/inspect_dataset.py --dataset RSHaze --data_dir /content/datasets/RSHaze
!python scripts/data/inspect_dataset.py --dataset RRSHID --data_dir /content/datasets/RRSHID


数据集分析：RSHazePlus
目录：/content/datasets/RSHazePlus

1. 目录结构分析
----------------------------------------
  子目录：['RSHaze+']

2. 图像文件统计
----------------------------------------
  总图像文件数：26760
  各目录文件数:
    cleanpng: 8060
    synhazypng: 8060
    airpng: 2660
    transpng: 2660
    nircleanpng: 1330
    nirtranspng: 1330
    nirairpng: 1330
    nirhazypng: 1330

3. 图像格式分析
----------------------------------------
  格式分布:
    PNG: 100
  颜色模式分布:
    RGB: 100
  分辨率分布 (前 10):
    512x512: 100
  抽样验证：100/100 有效

4. RGB 通道检查
----------------------------------------
  RGB/RGBA: 100
  Grayscale: 0
  Other: 0
  [OK] 数据集包含 RGB 图像

5. Hazy/Clear 配对检查
----------------------------------------
  可能的 Hazy 目录：['RSHaze+']
  可能的 Clear 目录：[]
  [INFO] 无法自动识别 hazy/clear 目录结构

6. 官方 Train/Test Split 检查
----------------------------------------
  发现的 Split 目录：[]
  [INFO] 未发现官方 split 划分，需要手动划分

7. 重复图像检查
----------------------------------------
  [WARN] 发现重复文件名：5130 个
    371.png: 14 次
    819.png: 14 次
    429.png: 

In [ ]:
!ls /content//datasets/RSHazePlus/RSHaze+/

RSHaze_G  RSHaze_L  RSHaze_S  SOTS


## 阶段 5B-1 完成：RSHaze+ Dataset Implementation

In [ ]:
# Step 1: 更新代码
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 3), reused 8 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 12.02 KiB | 3.00 MiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   891558f..6d7b65b  main       -> origin/main
Updating 891558f..6d7b65b
Fast-forward
 docs/project_status.md                            | 285 +++++++++++++++--
 docs/stage_5b2_report.md                          | 232 ++++++++++++++
 scripts/test_rshazeplus_physical_prior.py         | 355 ++++++++++++++++++++++
 scripts/visualize_physical_prior_on_rshazeplus.py | 259 ++++++++++++++++
 4 files changed, 1104 insertions(+), 27 deletions(-)
 create mode 100644 docs/stage_5b2_report.md
 create mode 100644 scripts/test_rshazeplus_physical_prior.py
 create mode 100644 scripts/visualize_physical_prior_on_rshazeplus.py


In [ ]:
# 生成 train/val/test split 文件
!python scripts/data/generate_rshazeplus_split.py
# 运行 Dataset 测试脚本
#!python scripts/test/test_rshazeplus_dataset.py
# 验证 train/val/test 无重叠
!python scripts/data/verify_rshazeplus_split.py
# 生成数据集预览图像
#!python scripts/data/visualize_rshazeplus_dataset.py


RSHaze+ Split 生成

扫描数据集...
[WARN] Subset directory not found: datasets/RSHaze+/RSHaze_G
[WARN] Subset directory not found: datasets/RSHaze+/RSHaze_L
[WARN] Subset directory not found: datasets/RSHaze+/RSHaze_S

官方 train 总计：0 samples
官方 test 总计：0 samples

RSHaze_G:
  官方 train: 0
  官方 test: 0
  Train: -1, Val: 1

RSHaze_L:
  官方 train: 0
  官方 test: 0
  Train: -1, Val: 1

RSHaze_S:
  官方 train: 0
  官方 test: 0
  Train: -1, Val: 1

Split 统计

Train: 0 samples
Val: 0 samples
Test: 0 samples

Train subset 分布:
  RSHaze_G: -1
  RSHaze_L: -1
  RSHaze_S: -1

Val subset 分布:
  RSHaze_G: 1
  RSHaze_L: 1
  RSHaze_S: 1

Test subset 分布:
  RSHaze_G: 0
  RSHaze_L: 0
  RSHaze_S: 0

保存 split 到：experiments/haze_density/rshazeplus_split.json

重叠检查
[OK] Train/Val: No overlap
[OK] Train/Test: No overlap
[OK] Val/Test: No overlap

内部重复检查:
[OK] Train: No internal duplicates
[OK] Val: No internal duplicates
[OK] Test: No internal duplicates

Split 生成完成

RSHaze+ Split 验证 (修复版)

JSON Schema

顶层 keys:
  train: list[0]

## 阶段 5B-2 准备完成

In [ ]:
# Step 1: 更新代码
!git pull

Already up to date.


In [ ]:
# 2. 运行 Integration 测试
!python scripts/test/test_rshazeplus_physical_prior.py
# 3. 生成可视化
!python scripts/data/visualize_physical_prior_on_rshazeplus.py


Stage 5B-2: RSHaze+ Dataset + Physical Prior Integration Test

Physical Prior Test (256x256)
Device: cuda
Train loader: 1543 batches
Batch 1: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0249, 0.9075], finite=True
Batch 2: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0419, 0.9355], finite=True
Batch 3: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0277, 0.8569], finite=True
Batch 4: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0410, 0.8993], finite=True
Batch 5: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0202, 0.8697], finite=True
Batch 6: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0434, 0.8274], finite=True
Batch 7: image torch.Size([4, 3, 256, 256]) -> S_final torch.Size([4, 1, 256, 256]), range [0.0348, 0.9601], finite=True
Batch 8: image torc

## 阶段 5C：HazeDensityNet 8-Image Overfit Test

In [ ]:
# Step 1: 更新代码
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 7.36 KiB | 1.23 MiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   6d7b65b..9b147ec  main       -> origin/main
Updating 6d7b65b..9b147ec
Fast-forward
 docs/project_status.md          |  47 ++--
 docs/stage_5c_overfit_report.md | 184 ++++++++++++++
 scripts/train_overfit_8.py      | 525 ++++++++++++++++++++++++++++++++++++++++
 3 files changed, 740 insertions(+), 16 deletions(-)
 create mode 100644 docs/stage_5c_overfit_report.md
 create mode 100644 scripts/train_overfit_8.py


In [ ]:
# 2. 运行训练
!python scripts/train/train_overfit_8.py


Stage 5C: HazeDensityNet 8-Image Overfit Test

Device: cuda
CUDA device: Tesla T4

Loading dataset...
Full train dataset: 6174 samples

Selecting 8 samples...
  RSHaze_G: selected 2 samples
  RSHaze_L: selected 4 samples
  RSHaze_S: selected 2 samples
Total selected: 8 samples
Saved sample list: experiments/haze_density/results/overfit_8_samples/sample_list.txt
DataLoader: 4 batches

Creating model...
HazeDensityNet Model Summary
base_channels: 32
use_sigmoid: True

Parameter Statistics:
  Encoder:    28,512 (3.3%)
  MultiScale: 700,416 (81.0%)
  Fusion:     122,880 (14.2%)
  Decoder:    12,944 (1.5%)
------------------------------------------------------------
  Total:      864,752
Saved model info: experiments/haze_density/results/overfit_8_samples/model_info.txt

Creating Physical Prior...
/content/CrossModal-Defog-OVOD/scripts/train_overfit_8.py:277: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  sca

## 阶段 5D-0: Formal Training Preflight Audit 准备完成

In [ ]:
# Step 1: 更新代码
!git pull

Already up to date.


In [3]:
# 运行 5 epochs smoke test
!python scripts/train/train_haze_density.py --epochs 5

python3: can't open file '/content/CrossModal-Defog-OVOD/scripts/train_haze_density.py': [Errno 2] No such file or directory


In [ ]:
# 训练 2 epochs
!python scripts/train/train_haze_density.py --epochs 2

# 恢复训练 (继续训练到 5 epochs)
!python scripts/train/train_haze_density.py --epochs 5 --resume experiments/haze_density/checkpoints/formal/latest.pth


Stage 5D: HazeDensityNet Formal Training

Device: cuda
CUDA device: Tesla T4

Creating model...
HazeDensityNet Model Summary
base_channels: 32
use_sigmoid: True

Parameter Statistics:
  Encoder:    28,512 (3.3%)
  MultiScale: 700,416 (81.0%)
  Fusion:     122,880 (14.2%)
  Decoder:    12,944 (1.5%)
------------------------------------------------------------
  Total:      864,752

Creating Physical Prior...

Loading datasets...
Train loader: 1543 batches (6174 samples)
Val loader: 172 batches (686 samples)
/content/CrossModal-Defog-OVOD/scripts/train_haze_density.py:326: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
AMP: Enabled

Training Start
/content/CrossModal-Defog-OVOD/scripts/train_haze_density.py:365: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch  1/2: 

## 阶段 5D-1: Prediction Distribution Audit 分析完成

In [ ]:
# Step 1: 更新代码
!git pull

Already up to date.


In [ ]:
# 重新运行 5 epochs smoke test
#!python scripts/train/train_haze_density.py --epochs 5

# 重新运行审计
!python scripts/evaluate/audit_prediction_distribution.py \
    --checkpoint experiments/haze_density/checkpoints/formal/best.pth


Stage 5D-1: Prediction Distribution Audit

Device: cuda

Loading checkpoint: experiments/haze_density/checkpoints/formal/best.pth
  Epoch: 2
  Val Loss: 0.010854

Creating model...
  base_channels: 32
  use_sigmoid: True

Creating Physical Prior...

Loading validation dataset...
Val loader: 179 batches (713 samples)

------------------------------------------------------------
Computing full statistics...
Total samples: 713
Shape per sample: torch.Size([1, 256, 256])

Prediction Distribution

Prediction Statistics:
  min: 0.000000
  max: 1.000000
  mean: 0.492274
  std: 0.168589
  p1: 0.134540
  p5: 0.234975
  p25: 0.370898
  p50: 0.477894
  p75: 0.611048
  p95: 0.773013
  p99: 0.933923
  <0.1: 0.45%
  <0.2: 3.01%
  <0.3: 11.73%
  <0.4: 32.11%
  <0.5: 54.48%

Target (S_final) Distribution

Target Statistics:
  min: 0.019632
  max: 0.976088
  mean: 0.452976
  std: 0.186229
  p1: 0.123832
  p5: 0.193024
  p25: 0.306159
  p50: 0.424492
  p75: 0.584665
  p95: 0.797797
  p99: 0.883071
  <0

In [ ]:
# 自动修复
!python scripts/utils/fix_decoder_relu.py

# 或手动修改 src/models/haze_density/decoder.py 第 94 行

Fix Decoder ReLU → Sigmoid Issue
Reading src/models/haze_density/decoder.py...
Backup saved: src/models/haze_density/decoder.py.bak

Fixed src/models/haze_density/decoder.py

Fixed code section:
------------------------------------------------------------
       85:         # Conv 2 (输出 1 通道)
 >>>   86:         self.conv2 = nn.Conv2d(
       87:             in_channels // 4,

       92:         )
 >>>   93:         self.norm3 = nn.InstanceNorm2d(1)
       94:         # 【修复】移除 ReLU，避免 ReLU → Sigmoid 导致输出范围 [0.5, 1)

       94:         # 【修复】移除 ReLU，避免 ReLU → Sigmoid 导致输出范围 [0.5, 1)
 >>>   95:         self.relu3 = nn.Identity()  # 原为 nn.ReLU(inplace=True)
       96: 

      118:         # Conv 2
 >>>  119:         x = self.conv2(x)  # [B, 1, H, W]
      120:         x = self.norm3(x)

      119:         x = self.conv2(x)  # [B, 1, H, W]
 >>>  120:         x = self.norm3(x)
      121:         x = self.relu3(x)

      120:         x = self.norm3(x)
 >>>  121:         x = self.relu3(x)
    

In [ ]:
!python scripts/train/train_haze_density.py --epochs 5


Stage 5D: HazeDensityNet Formal Training

Device: cuda
CUDA device: Tesla T4

Creating model...
HazeDensityNet Model Summary
base_channels: 32
use_sigmoid: True

Parameter Statistics:
  Encoder:    28,512 (3.3%)
  MultiScale: 700,416 (81.0%)
  Fusion:     122,880 (14.2%)
  Decoder:    12,944 (1.5%)
------------------------------------------------------------
  Total:      864,752

Creating Physical Prior...

Loading datasets...
Train loader: 1604 batches (6417 samples)
Val loader: 179 batches (713 samples)
/content/CrossModal-Defog-OVOD/scripts/train_haze_density.py:326: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
AMP: Enabled

Training Start
/content/CrossModal-Defog-OVOD/scripts/train_haze_density.py:365: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch  1/5: 

## 阶段 5D-2: Decoder Activation Verification 完成

In [ ]:
# Step 0: 更新代码
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 5.69 KiB | 2.84 MiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   f89769f..3ab875b  main       -> origin/main
Updating f89769f..3ab875b
Fast-forward
 scripts/evaluate_haze_density.py |  26 +++--
 scripts/train_haze_density.py    |  47 ++++-----
 src/utils/path_utils.py          | 200 ---------------------------------------
 3 files changed, 32 insertions(+), 241 deletions(-)
 delete mode 100644 src/utils/path_utils.py


In [ ]:
# 步骤 1: 重新训练 5 epochs
!python scripts/train/train_haze_density.py --epochs 5


Stage 5D: HazeDensityNet Formal Training

Device: cuda
CUDA device: Tesla T4
[DEBUG] Detected Colab via google.colab.runtime
[INFO] 检测到环境：colab
[INFO] 数据集路径：/content/datasets/RSHazePlus/RSHaze+

Dataset root: /content/datasets/RSHazePlus/RSHaze+
Split file: experiments/haze_density/rshazeplus_split.json
Checkpoint dir: experiments/haze_density/checkpoints/formal
Result dir: experiments/haze_density/results

Creating model...
HazeDensityNet Model Summary
base_channels: 32
use_sigmoid: True

Parameter Statistics:
  Encoder:    28,512 (3.3%)
  MultiScale: 700,416 (81.0%)
  Fusion:     122,880 (14.2%)
  Decoder:    12,944 (1.5%)
------------------------------------------------------------
  Total:      864,752

Creating Physical Prior...

Loading datasets...
[WARN] Split file experiments/haze_density/rshazeplus_split.json has unexpected format, using default split
[WARN] Split file experiments/haze_density/rshazeplus_split.json has unexpected format, using default split
Train loader: 1782

In [ ]:
# 步骤 2: 运行 audit 脚本验证
!python scripts/evaluate/audit_prediction_distribution.py \
    --checkpoint experiments/haze_density/checkpoints/formal/best.pth


Stage 5D-1: Prediction Distribution Audit

Device: cuda

Loading checkpoint: experiments/haze_density/checkpoints/formal/best.pth
  Epoch: 5
  Val Loss: 0.009657

Creating model...
  base_channels: 32
  use_sigmoid: True

Creating Physical Prior...

Loading validation dataset...
Val loader: 179 batches (713 samples)

------------------------------------------------------------
Computing full statistics...
Total samples: 713
Shape per sample: torch.Size([1, 256, 256])

Prediction Distribution

Prediction Statistics:
  min: 0.000000
  max: 1.000000
  mean: 0.489749
  std: 0.170277
  p1: 0.149222
  p5: 0.240988
  p25: 0.368962
  p50: 0.466365
  p75: 0.602731
  p95: 0.789913
  p99: 0.961283
  <0.1: 0.35%
  <0.2: 2.51%
  <0.3: 11.62%
  <0.4: 32.84%
  <0.5: 57.59%

Target (S_final) Distribution

Target Statistics:
  min: 0.019632
  max: 0.976088
  mean: 0.452976
  std: 0.186229
  p1: 0.123832
  p5: 0.193024
  p25: 0.306159
  p50: 0.424492
  p75: 0.584665
  p95: 0.797797
  p99: 0.883071
  <0

In [ ]:
# 步骤 3: 检查验收标准
!cat experiments/haze_density/results/formal_prediction_audit/audit_report.txt

Stage 5D-1: Prediction Distribution Audit Report

Checkpoint: experiments/haze_density/checkpoints/formal/best.pth
Epoch: 5
Val Loss: 0.009657

Prediction Statistics:
  min: 0.000000
  max: 1.000000
  mean: 0.489749
  std: 0.170277
  p1: 0.149222
  p5: 0.240988
  p25: 0.368962
  p50: 0.466365
  p75: 0.602731
  p95: 0.789913
  p99: 0.961283
  <0.1: 0.350310
  <0.2: 2.506664
  <0.3: 11.615970
  <0.4: 32.838556
  <0.5: 57.587661

Target Statistics:
  min: 0.019632
  max: 0.976088
  mean: 0.452976
  std: 0.186229
  p1: 0.123832
  p5: 0.193024
  p25: 0.306159
  p50: 0.424492
  p75: 0.584665
  p95: 0.797797
  p99: 0.883071
  <0.1: 0.336059
  <0.2: 5.666892
  <0.3: 23.601910
  <0.4: 45.264305
  <0.5: 62.997792

Pearson correlation: 0.874913
MAE: 0.078139
RMSE: 0.098270
MSE: 0.009657

Conclusion: A


## 阶段 5E: Test Evaluation 脚本创建完成

In [ ]:
# Step 0: 更新代码
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 416 bytes | 208.00 KiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   803565f..f17f4c5  main       -> origin/main
Updating 803565f..f17f4c5
Fast-forward
 scripts/evaluate_haze_density.py | 4 ++++
 1 file changed, 4 insertions(+)


In [ ]:
# 运行测试评估
!python scripts/evaluate/evaluate_haze_density.py \
    --checkpoint experiments/haze_density/checkpoints/formal/best.pth \
    --image_size 256 \
    --batch_size 4 \
    --num_samples_per_subset 16


Stage 5E: HazeDensityNet Test Evaluation

Device: cuda

Loading checkpoint: experiments/haze_density/checkpoints/formal/best.pth
  Epoch: 5
  Val Loss: 0.009657

Creating model...
  base_channels: 32
  use_sigmoid: True

Creating Physical Prior...

Loading test dataset...
Test loader: 233 batches (930 samples)

Test dataset statistics:
  Total samples: 930
  Subset counts: {'RSHaze_G': 330, 'RSHaze_L': 270, 'RSHaze_S': 330}

Evaluation

Total test samples: 930
Shape per sample: torch.Size([1, 256, 256])
[OK] No NaN detected
Prediction range: [0.0000, 1.0000]
[OK] Prediction range within [0, 1]

Overall Metrics

Overall Test Metrics:
  MSE:     0.009809
  MAE:     0.079607
  RMSE:    0.099042
  Pearson: 0.872662

Metrics by Subset

RSHaze_G (330 samples):
  MAE:     0.079406
  RMSE:    0.099076
  Pearson: 0.876442

RSHaze_L (270 samples):
  MAE:     0.077172
  RMSE:    0.097134
  Pearson: 0.875329

RSHaze_S (330 samples):
  MAE:     0.081800
  RMSE:    0.100544
  Pearson: 0.869471

Pre

In [ ]:
# 查看结果
!cat experiments/haze_density/results/test_evaluation/test_metrics.txt

Stage 5E: HazeDensityNet Test Evaluation Report

Checkpoint Information:
  Path: experiments/haze_density/checkpoints/formal/best.pth
  Epoch: 5
  Val Loss: 0.009657

Dataset Information:
  Total test samples: 930
  Subset counts: {'RSHaze_G': 330, 'RSHaze_L': 270, 'RSHaze_S': 330}

Overall Metrics:
  MSE: 0.009809
  MAE: 0.079607
  RMSE: 0.099042
  Pearson: 0.872662

Metrics by Subset:

RSHaze_G:
  MSE: 0.009816
  MAE: 0.079406
  RMSE: 0.099076
  Pearson: 0.876442

RSHaze_L:
  MSE: 0.009435
  MAE: 0.077172
  RMSE: 0.097134
  Pearson: 0.875329

RSHaze_S:
  MSE: 0.010109
  MAE: 0.081800
  RMSE: 0.100544
  Pearson: 0.869471

Prediction Distribution:
  mean: 0.487148
  std: 0.159688
  min: 0.000000
  max: 1.000000
  p5: 0.258219
  p25: 0.378553
  p50: 0.463353
  p75: 0.582899
  p95: 0.774726

Target Distribution:
  mean: 0.435942
  std: 0.170349
  min: 0.020436
  max: 0.994765
  p5: 0.195202
  p25: 0.306705
  p50: 0.409753
  p75: 0.550181
  p95: 0.752657


In [ ]:
# 列出可视化文件
!ls -la experiments/haze_density/results/test_evaluation/

total 12800
drwxr-xr-x 2 root root   4096 Sep  2 02:59 .
drwxr-xr-x 5 root root   4096 Sep  2 02:41 ..
-rw-r--r-- 1 root root 316324 Sep  2 02:57 RSHaze_G_1013.png
-rw-r--r-- 1 root root 255745 Sep  2 02:57 RSHaze_G_1016.png
-rw-r--r-- 1 root root 280098 Sep  2 02:57 RSHaze_G_1017.png
-rw-r--r-- 1 root root 316578 Sep  2 02:57 RSHaze_G_1045.png
-rw-r--r-- 1 root root 302071 Sep  2 02:57 RSHaze_G_1048.png
-rw-r--r-- 1 root root 316131 Sep  2 02:57 RSHaze_G_1053.png
-rw-r--r-- 1 root root 319352 Sep  2 02:57 RSHaze_G_1058.png
-rw-r--r-- 1 root root 228304 Sep  2 02:57 RSHaze_G_1072.png
-rw-r--r-- 1 root root 320258 Sep  2 02:57 RSHaze_G_1112.png
-rw-r--r-- 1 root root 303895 Sep  2 02:57 RSHaze_G_1115.png
-rw-r--r-- 1 root root 275419 Sep  2 02:57 RSHaze_G_1126.png
-rw-r--r-- 1 root root 336597 Sep  2 02:57 RSHaze_G_1141.png
-rw-r--r-- 1 root root 301194 Sep  2 02:57 RSHaze_G_1217.png
-rw-r--r-- 1 root root 307315 Sep  2 02:57 RSHaze_G_1280.png
-rw-r--r-- 1 root root 290691 Sep  2 02:57 

## 阶段 6-1: HazeDensityNet Integration Planning 完成

## 阶段 6-2: HazeDensityNet Baseline Integration 完成

In [ ]:
# Step 0: 更新代码
!git pull

Already up to date.


In [ ]:
# 运行 integration test
!python scripts/test/test_density_concat_integration.py


Stage 6-2: Density Concatenation Integration Test

Test 1: Forward Shape Test
Device: cpu

Creating model...
[DensityConcatModel] Loading HazeDensityNet from experiments/haze_density/checkpoints/formal/best.pth
[DensityConcatModel] Loaded density checkpoint successfully
[DensityConcatModel] HazeDensityNet frozen (requires_grad=False)
[DensityConcatModel] Creating 4-channel backbone

[DensityConcatModel] Parameter Statistics:
  Total parameters:      7,148,144
  HazeDensityNet:        864,752 (frozen=True)
  Backbone:              6,283,392
  Trainable parameters:  6,283,392


Test cases:
  Batch=1, Size=256: PASS
    RGB:        (1, 3, 256, 256)
    Density:    (1, 1, 256, 256)
    Concat:     (1, 4, 256, 256)
    Features:   [(1, 128, 128, 128), (1, 256, 64, 64), (1, 512, 32, 32), (1, 1024, 16, 16)]
  Batch=2, Size=256: PASS
    RGB:        (2, 3, 256, 256)
    Density:    (2, 1, 256, 256)
    Concat:     (2, 4, 256, 256)
    Features:   [(2, 128, 128, 128), (2, 256, 64, 64), (2, 512

## 阶段 6-3: Attention Guidance

In [ ]:
# Step 0: 更新代码
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 512 bytes | 512.00 KiB/s, done.
From https://github.com/codeGease-1/CrossModal-Defog-OVOD
   8b7b7ab..6c72da7  main       -> origin/main
Updating 8b7b7ab..6c72da7
Fast-forward
 src/models/density_guidance/__init__.py | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)


In [ ]:
!python3 scripts/test/test_density_guidance.py


Stage 6-3A: Density Guidance Module Test

Test 1: Shape Test (F0/F1/F2/F3)
Device: cpu

Test cases (density_size=256):
  F0: PASS
    Input feature:  (2, 128, 128, 128)
    Density map:    (2, 1, 256, 256)
    Output feature: (2, 128, 128, 128)
  F1: PASS
    Input feature:  (2, 256, 64, 64)
    Density map:    (2, 1, 256, 256)
    Output feature: (2, 256, 64, 64)
  F2: PASS
    Input feature:  (2, 512, 32, 32)
    Density map:    (2, 1, 256, 256)
    Output feature: (2, 512, 32, 32)
  F3: PASS
    Input feature:  (2, 1024, 16, 16)
    Density map:    (2, 1, 256, 256)
    Output feature: (2, 1024, 16, 16)

[OK] Shape test passed

Test 2: Identity Initialization Test
Device: cpu

Gamma initial value: 0.0000000000

Difference (guided - feature):
  Max diff:  0.0000000000e+00
  Mean diff: 0.0000000000e+00

[OK] Identity initialization test passed

Test 3: Gradient Flow Test
Device: cpu

Gradient status:
  visual_feature has grad: True (should be True)
  module params have grad: True (sho